In [ ]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import src.config as cfg
from src.data_loader import load_bars, split_train_test
from src.indicators import add_indicators
from src.strategy import generate_signals
from src.backtest import run_backtest
from src.reporting import save_iteration, run_monte_carlo
import json

VERSION = "V1"
DATA_PATH = "data/NQ_1min.csv"


In [ ]:
df = load_bars(DATA_PATH)
train, test = split_train_test(df, cfg.TRAIN_RATIO)
print(f"Total: {len(df):,}  Train: {len(train):,}  Test: {len(test):,}")


In [ ]:
train = add_indicators(train, cfg)
print("Indicator columns:", [c for c in train.columns if c not in ['time','open','high','low','close','volume','session_break']])
train.head(3)


In [ ]:
train = generate_signals(train, cfg)
n_long = (train.signal == 1).sum()
n_short = (train.signal == -1).sum()
print(f"Signals — long: {n_long:,}  short: {n_short:,}")


In [ ]:
results = run_backtest(train, cfg, version=VERSION)
trades = results["trades"]
n = results["n_trades"]
wins = sum(1 for t in trades if t["label_win"])
print(f"Trades: {n:,}  Wins: {wins:,}  Win rate: {wins/n*100:.1f}%  Final equity: ${results['final_equity']:,.2f}")


In [ ]:
out_dir = save_iteration(VERSION, results, train, cfg)
print(f"Artifacts saved to: {out_dir}")


In [ ]:
mc_path = out_dir / "monte_carlo.json"
with open(mc_path) as f:
    mc = json.load(f)
print(f"Monte Carlo ({mc['n_sims']:,} sims, seed={mc['seed']})")
print(f"  Max Drawdown p50: ${mc['max_drawdown']['p50']:.2f}  p90: ${mc['max_drawdown']['p90']:.2f}  p99: ${mc['max_drawdown']['p99']:.2f}")
print(f"  VaR (5th pct): ${mc['var_trade_pnl']:.2f}  CVaR: ${mc['cvar_trade_pnl']:.2f}")
print(f"  Risk of ruin: {mc['risk_of_ruin_prob']*100:.1f}%")


In [ ]:
tlog = pd.read_csv(out_dir / "trader_log.csv")

def color_rows(row):
    if row["net_pnl_dollars"] > 0:
        return ["background-color: #d4edda"] * len(row)
    elif row["net_pnl_dollars"] < 0:
        return ["background-color: #f8d7da"] * len(row)
    else:
        return [""] * len(row)

styled = tlog.head(50).style.apply(color_rows, axis=1)
styled


In [ ]:
html_path = out_dir / "trader_log.html"
tlog.head(200).style.apply(color_rows, axis=1).to_html(html_path)
print(f"Styled HTML saved: {html_path}")
